In [2]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()


In [ ]:
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage, AssistantMessage
from gen_ai_hub.orchestration.models.template import Template, TemplateValue
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.response_format import ResponseFormatJsonSchema 
from gen_ai_hub.orchestration.models.llm import LLM
import json
import requests
from datetime import datetime

In [ ]:
llm = LLM(name="gpt-4o", version="latest", parameters={"max_tokens": 256, "temperature": 0.2})

config = OrchestrationConfig(
    template=Template(
        messages=[
            SystemMessage("You are a helpful AI Assistent that answer the best you can all the time."),
            UserMessage("{{?question}}"),
        ]
    ),
    llm=llm    
)

In [23]:
orchestration_service = OrchestrationService(config=config)

def send_request(config, **kwargs):
    template_values = [TemplateValue(name=key, value=value) for key, value in kwargs.items()]
    answer = orchestration_service.run(config=config, template_values=template_values)
    return answer.module_results.llm.choices[0].message.content 

In [ ]:
result1 = send_request(config, question = "What time is it now?")
print(result1)
result2 = send_request(config, question = "How is the weather there now?")
print(result2)

I'm sorry, but I don't have real-time capabilities to provide the current time. You can check the time on your device or a clock nearby.
I'm sorry, but I can't provide real-time weather updates. To find the current weather in Beijing, I recommend checking a reliable weather website or app, such as the Weather Channel, BBC Weather, or a local news source.


In [27]:
questions = [
    "Which city is the capital of China?",
    "How is the weather there now?",
    "What time is it now?"
]


for q in questions:
    result = send_request(config, question=q)
    print(f"Question: {q}")
    print(f"Answer: {result}")
    print("-" * 40)  # Separator for readabilit


Question: Which city is the capital of China?
Answer: The capital of China is Beijing.
----------------------------------------
Question: How is the weather there now?
Answer: I'm sorry, but I don't have real-time capabilities to check the current weather. You might want to use a weather website or app to get the latest updates for your location.
----------------------------------------
Question: What time is it now?
Answer: I'm sorry, but I can't provide real-time information, including the current time. You can check the time on your device or a clock nearby.
----------------------------------------


In [28]:

def get_time_now():
    """Returns the current local time as a formatted string."""
    return {"time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

In [29]:

def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]

In [30]:
get_time_now()

{'time': '2025-12-10 07:46:11'}

In [31]:
result = send_request(config, question = "What is latitude and longitude of Beijing?")
print(result)

The latitude and longitude of Beijing, China, are approximately 39.9042° N (latitude) and 116.4074° E (longitude).


In [32]:
get_weather(39.9042, 116.4074)

{'time': '2025-12-10T07:45',
 'interval': 900,
 'temperature_2m': 4.0,
 'wind_speed_10m': 3.7}

In [33]:

class ToolRegistry:
    def __init__(self):
        self.tools = {}

    def register(self, name, function, description, parameters):
        self.tools[name] = {
            "function": function,
            "description": description,
            "parameters": parameters
        }

    def get_description_for_prompt(self):
        return {
            name: {
                "description": entry["description"],
                "parameters": entry["parameters"]
            } for name, entry in self.tools.items()
        }

    def get_callable(self, name):
        return self.tools.get(name, {}).get("function")

In [34]:
registry = ToolRegistry()
registry.register(
    "get_weather",
    get_weather,
    "Retrieves current weather data for a given set of geographic coordinates (latitude, longitude).",
    {
        "latitude": "float - The latitude of the location.",
        "longitude": "float - The longitude of the location."
    }
)

registry.register(
    "get_time_now",
    get_time_now,
    "Returns the current local time in YYYY-MM-DD HH:MM:SS format.",
    {}  # No parameters required
)


In [35]:

tool_prompt = json.dumps(registry.get_description_for_prompt(), indent=2)
# 这个 dict 通常会做成 JSON/文本放进系统提示里
print(tool_prompt)

{
  "get_weather": {
    "description": "Retrieves current weather data for a given set of geographic coordinates (latitude, longitude).",
    "parameters": {
      "latitude": "float - The latitude of the location.",
      "longitude": "float - The longitude of the location."
    }
  },
  "get_time_now": {
    "description": "Returns the current local time in YYYY-MM-DD HH:MM:SS format.",
    "parameters": {}
  }
}


In [36]:
import AgentExecutor

In [42]:
agent = AgentExecutor.AgentExecutor(llm=llm, tool_registry=registry, verbose=True)


prompt = """Can you tell me:

1. What is the capital of China?
2. How is the weather there now?
3. What time is it now?
"""

response = agent.run(prompt)
print("\n",response)


LLM Reasoning:
{
  "tool_calls": [
    {
      "decision": "tool",
      "reason": "The user asked for weather in Beijing, China.",
      "function": "get_weather",
      "parameters": {
        "latitude": 39.9042,
        "longitude": 116.4074
      }
    },
    {
      "decision": "tool",
      "reason": "The user asked for the current time.",
      "function": "get_time_now",
      "parameters": {}
    }
  ]
}

Tool 'get_weather' executed with args {'latitude': 39.9042, 'longitude': 116.4074}. Result: {'time': '2025-12-10T07:45', 'interval': 900, 'temperature_2m': 4.0, 'wind_speed_10m': 3.7}

Tool 'get_time_now' executed with args {}. Result: {'time': '2025-12-10 07:52:19'}

 1. The capital of China is Beijing.

2. The current weather in Beijing is 4°C with a wind speed of 3.7 m/s.

3. The current local time is 07:52 AM on December 10, 2025.
